In [ ]:
import torch

print("PyTorch版本：", torch.__version__)
print("CUDA是否可用：", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU型号：", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("没有检测到GPU！请重新选择 T4 GPU 运行时。")

In [ ]:
!pip install -U ultralytics -q

In [ ]:
import ultralytics
from ultralytics import YOLO

print("Ultralytics版本：", ultralytics.__version__)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path

drive_root = Path("/content/drive/MyDrive")

yaml_files = list(drive_root.rglob("*.yaml")) + list(drive_root.rglob("*.yml"))

print(f"共找到 {len(yaml_files)} 个YAML文件：\n")

for i, path in enumerate(yaml_files):
    print(f"[{i}] {path}")

In [ ]:
from pathlib import Path

V9_DATA_YAML = Path(
    "/content/drive/MyDrive/fire_smoke_project/"
    "runs/yolov9s_formal_100e/data_colab_used.yaml"
)

V11_DATA_YAML = Path(
    "/content/drive/MyDrive/fire_smoke_project/"
    "runs/yolo11s_640_100e_b16_s42/data_colab_used.yaml"
)

print("YOLOv9数据配置是否存在：", V9_DATA_YAML.exists())
print("YOLO11数据配置是否存在：", V11_DATA_YAML.exists())

print("\n========== YOLOv9数据配置 ==========")
print(V9_DATA_YAML.read_text(encoding="utf-8"))

print("\n========== YOLO11数据配置 ==========")
print(V11_DATA_YAML.read_text(encoding="utf-8"))

In [ ]:
import yaml
from pprint import pprint

with open(V9_DATA_YAML, "r", encoding="utf-8") as f:
    v9_data = yaml.safe_load(f)

with open(V11_DATA_YAML, "r", encoding="utf-8") as f:
    v11_data = yaml.safe_load(f)

print("========== YOLOv9配置 ==========")
pprint(v9_data)

print("\n========== YOLO11配置 ==========")
pprint(v11_data)

print("\n========== 关键项目对比 ==========")

for key in ["path", "train", "val", "test", "names", "nc"]:
    print(f"\n{key}:")
    print("YOLOv9 ：", v9_data.get(key))
    print("YOLO11：", v11_data.get(key))

In [ ]:
DATA_YAML = str(V11_DATA_YAML)

print("YOLOv8-S将使用的数据配置：")
print(DATA_YAML)

In [ ]:
!pip install -U kagglehub -q

print("kagglehub 安装完成")

In [ ]:
import kagglehub
from pathlib import Path

# Restore the dataset to the expected working directory.
DOWNLOAD_DIR = Path("/content/datasets/fire_smoke")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

print("开始下载数据集，请耐心等待……")

downloaded_path = kagglehub.dataset_download(
    "sayedgamal99/smoke-fire-detection-yolo",
    output_dir=str(DOWNLOAD_DIR)
)

print("\n数据集下载完成！")
print("Kaggle返回的保存位置：")
print(downloaded_path)

In [ ]:
from pathlib import Path
import kagglehub
import yaml

KAGGLE_DATASET = "sayedgamal99/smoke-fire-detection-yolo"

# --------------------------------------------------
# 1. Resolve the Kaggle download path.
# --------------------------------------------------
try:
    REAL_DOWNLOAD_PATH = Path(downloaded_path)
    print("使用刚才Kaggle返回的下载路径。")
except NameError:
    print("没有找到downloaded_path变量，重新获取数据集路径……")
    REAL_DOWNLOAD_PATH = Path(
        kagglehub.dataset_download(KAGGLE_DATASET)
    )

print("\nKaggle实际返回路径：")
print(REAL_DOWNLOAD_PATH)
print("该路径是否存在：", REAL_DOWNLOAD_PATH.exists())


# --------------------------------------------------
# 2. Check whether a directory contains a complete YOLO dataset.
# --------------------------------------------------
def is_complete_yolo_dataset(root):
    required_paths = [
        root / "train" / "images",
        root / "train" / "labels",
        root / "val" / "images",
        root / "val" / "labels",
        root / "test" / "images",
        root / "test" / "labels",
    ]

    return all(path.is_dir() for path in required_paths)


# --------------------------------------------------
# 3. Locate the dataset root automatically.
# --------------------------------------------------
def find_dataset_roots(search_root):
    candidates = []

    if not search_root.exists():
        return candidates

    # Common candidate locations.
    possible_roots = [
        search_root,
        search_root / "data",
    ]

    # Search for image directories.
    for images_folder in search_root.rglob("images"):
        if (
            images_folder.is_dir()
            and images_folder.parent.name == "train"
        ):
            possible_roots.append(
                images_folder.parent.parent
            )

    # Remove duplicate candidates and verify completeness.
    seen = set()

    for root in possible_roots:
        root_string = str(root.resolve())

        if root_string in seen:
            continue

        seen.add(root_string)

        if is_complete_yolo_dataset(root):
            candidates.append(root)

    return candidates


dataset_roots = find_dataset_roots(REAL_DOWNLOAD_PATH)


# --------------------------------------------------
# 4. Fall back to the Kaggle cache if required.
# --------------------------------------------------
if not dataset_roots:
    print("\n第一次没有找到完整目录。")
    print("现在重新获取Kaggle默认缓存路径，请稍等……")

    REAL_DOWNLOAD_PATH = Path(
        kagglehub.dataset_download(
            KAGGLE_DATASET,
            force_download=True
        )
    )

    print("\n重新下载后返回的路径：")
    print(REAL_DOWNLOAD_PATH)
    print("路径是否存在：", REAL_DOWNLOAD_PATH.exists())

    dataset_roots = find_dataset_roots(
        REAL_DOWNLOAD_PATH
    )


# --------------------------------------------------
# 5. Verify the resolved paths.
# --------------------------------------------------
if not dataset_roots:
    print("\n仍然没有识别到完整数据集。")
    print("下载目录中的前80个文件或文件夹：\n")

    if REAL_DOWNLOAD_PATH.exists():
        for index, item in enumerate(
            REAL_DOWNLOAD_PATH.rglob("*")
        ):
            print(item)

            if index >= 79:
                break

    raise FileNotFoundError(
        "没有找到完整的train/val/test数据结构。"
    )


ACTUAL_DATASET_ROOT = dataset_roots[0]

print("\n✅ 已找到完整数据集！")
print("数据集根目录：")
print(ACTUAL_DATASET_ROOT)


# --------------------------------------------------
# 6. Preserve the original dataset class order.
# --------------------------------------------------
with open(
    V9_DATA_YAML,
    "r",
    encoding="utf-8"
) as file:
    old_config = yaml.safe_load(file)

class_names = old_config["names"]

print("\n沿用以前的类别顺序：")
print(class_names)


# --------------------------------------------------
# 7. Generate a Colab-specific dataset YAML for YOLOv8-S.
# --------------------------------------------------
NEW_DATA_YAML = Path(
    "/content/drive/MyDrive/fire_smoke_project/"
    "runs/yolov8s_data_colab_used.yaml"
)

new_config = {
    "train": str(
        ACTUAL_DATASET_ROOT / "train" / "images"
    ),
    "val": str(
        ACTUAL_DATASET_ROOT / "val" / "images"
    ),
    "test": str(
        ACTUAL_DATASET_ROOT / "test" / "images"
    ),
    "names": class_names,
}

with open(
    NEW_DATA_YAML,
    "w",
    encoding="utf-8"
) as file:
    yaml.safe_dump(
        new_config,
        file,
        sort_keys=False,
        allow_unicode=True
    )

# Use this configuration for the remaining training cells.
DATA_YAML = str(NEW_DATA_YAML)

print("\n✅ 新的数据配置文件已经生成：")
print(DATA_YAML)

print("\n配置文件内容：")
print(
    NEW_DATA_YAML.read_text(
        encoding="utf-8"
    )
)


# --------------------------------------------------
# 8. Final dataset check.
# --------------------------------------------------
print("========== 最终路径检查 ==========")

all_paths_ok = True

for split in ["train", "val", "test"]:
    images_path = Path(new_config[split])
    labels_path = (
        ACTUAL_DATASET_ROOT
        / split
        / "labels"
    )

    print(f"\n{split.upper()}：")
    print("图片目录：", images_path)
    print("图片存在：", images_path.is_dir())
    print("标签目录：", labels_path)
    print("标签存在：", labels_path.is_dir())

    if not images_path.is_dir():
        all_paths_ok = False

    if not labels_path.is_dir():
        all_paths_ok = False


if all_paths_ok:
    print("\n✅ 数据集已经准备好，可以进行1轮测试训练。")
else:
    raise RuntimeError(
        "部分图片或标签路径仍然不存在。"
    )

In [ ]:
from pathlib import Path
import yaml

with open(DATA_YAML, "r", encoding="utf-8") as f:
    dataset_config = yaml.safe_load(f)

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp",
    ".webp", ".tif", ".tiff"
}

def count_images(folder):
    folder = Path(folder)
    return sum(
        1 for file in folder.rglob("*")
        if file.is_file()
        and file.suffix.lower() in IMAGE_EXTENSIONS
    )

def count_labels(folder):
    folder = Path(folder)
    return sum(
        1 for file in folder.rglob("*.txt")
        if file.is_file()
    )

print("========== 数据集数量检查 ==========\n")

for split in ["train", "val", "test"]:
    images_path = Path(dataset_config[split])
    labels_path = images_path.parent / "labels"

    image_count = count_images(images_path)
    label_count = count_labels(labels_path)

    print(f"{split.upper()}")
    print("图片数量：", image_count)
    print("标签数量：", label_count)
    print("-" * 40)

In [ ]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/fire_smoke_project/runs"
)

RUN_NAME = "yolov8s_640_100e_b16_s42"

RUN_DIR = PROJECT_DIR / RUN_NAME

PROJECT_DIR.mkdir(parents=True, exist_ok=True)

print("结果总目录：")
print(PROJECT_DIR)

print("\n正式训练目录：")
print(RUN_DIR)

print("\n正式训练目录是否已经存在：")
print(RUN_DIR.exists())

In [ ]:
from ultralytics import YOLO

preflight_model = YOLO("yolov8s.pt")

print("YOLOv8-S预训练模型加载成功！")
preflight_model.info()

In [ ]:
from ultralytics import YOLO

preflight_model = YOLO("yolov8s.pt")

preflight_results = preflight_model.train(
    data=DATA_YAML,

    # Short smoke test.
    epochs=1,

    # Formal experiment parameters.
    imgsz=640,
    batch=16,
    seed=42,
    deterministic=True,

    # Use the Colab GPU.
    device=0,
    workers=2,

    # Training configuration.
    pretrained=True,
    optimizer="auto",
    amp=True,
    cache=False,

    # Save smoke-test outputs.
    project=str(PROJECT_DIR),
    name="yolov8s_preflight_1e",
    exist_ok=True,

    plots=True,
    verbose=True
)

In [ ]:
from pathlib import Path

PREFLIGHT_DIR = (
    PROJECT_DIR / "yolov8s_preflight_1e"
)

PREFLIGHT_BEST = (
    PREFLIGHT_DIR / "weights" / "best.pt"
)

PREFLIGHT_LAST = (
    PREFLIGHT_DIR / "weights" / "last.pt"
)

print("试运行文件夹：", PREFLIGHT_DIR)
print("试运行best.pt存在：", PREFLIGHT_BEST.exists())
print("试运行last.pt存在：", PREFLIGHT_LAST.exists())

if PREFLIGHT_LAST.exists():
    print("\n✅ 1轮试训练成功，可以开始正式100轮训练。")
else:
    raise RuntimeError(
        "试运行没有正常生成last.pt，暂时不要正式训练。"
    )

In [ ]:
from ultralytics import YOLO
from pathlib import Path

RUN_DIR = PROJECT_DIR / RUN_NAME

# Avoid overwriting an existing formal run.
if RUN_DIR.exists():
    raise FileExistsError(
        f"正式训练目录已经存在：{RUN_DIR}\n"
        "请不要直接覆盖。"
    )

# Start from the official YOLOv8-S pretrained weights.
model = YOLO("yolov8s.pt")

results = model.train(
    data=DATA_YAML,

    # Common training parameters.
    epochs=100,
    imgsz=640,
    batch=16,
    seed=42,
    deterministic=True,

    # Colab L4 GPU
    device=0,
    workers=2,

    # Training configuration.
    pretrained=True,
    optimizer="auto",
    amp=True,
    cache=False,

    # Disable early stopping for the formal run.
    patience=100,

    # Save model weights and training plots.
    save=True,
    save_period=10,
    val=True,
    plots=True,
    verbose=True,

    # Save outputs to Google Drive.
    project=str(PROJECT_DIR),
    name=RUN_NAME,
    exist_ok=False
)

In [ ]:
%pip install -q ultralytics==8.4.115

In [ ]:
import ultralytics
print("Ultralytics 安装成功，版本：", ultralytics.__version__)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path
from ultralytics import YOLO

RUN_DIR = Path(
    "/content/drive/MyDrive/fire_smoke_project/"
    "runs/yolov8s_640_100e_b16_s42"
)

BEST_PT = RUN_DIR / "weights" / "best.pt"
LAST_PT = RUN_DIR / "weights" / "last.pt"

print("=" * 60)
print("训练结果文件夹：", RUN_DIR)
print("训练文件夹存在：", RUN_DIR.exists())
print()

print("best.pt 路径：", BEST_PT)
print("best.pt 存在：", BEST_PT.exists())

if BEST_PT.exists():
    print(
        "best.pt 大小：",
        round(BEST_PT.stat().st_size / 1024 / 1024, 2),
        "MB"
    )

print()
print("last.pt 存在：", LAST_PT.exists())
print("=" * 60)

assert RUN_DIR.exists(), f"找不到训练结果文件夹：{RUN_DIR}"
assert BEST_PT.exists(), f"找不到 best.pt：{BEST_PT}"

print("\n开始加载 best.pt……")

model = YOLO(str(BEST_PT))

print("\n模型加载成功！")
print("模型类别：", model.names)

In [ ]:
from pathlib import Path
from shutil import copy2
from google.colab import files

BEST_PT = Path(
    "/content/drive/MyDrive/fire_smoke_project/"
    "runs/yolov8s_640_100e_b16_s42/weights/best.pt"
)

assert BEST_PT.exists(), f"找不到模型文件：{BEST_PT}"

DOWNLOAD_MODEL = Path("/content/yolov8s_best.pt")

copy2(BEST_PT, DOWNLOAD_MODEL)

print("模型已经复制到 Colab 临时下载目录：")
print(DOWNLOAD_MODEL)
print(
    "模型大小：",
    round(DOWNLOAD_MODEL.stat().st_size / 1024 / 1024, 2),
    "MB"
)

files.download(str(DOWNLOAD_MODEL))

In [ ]:
from pathlib import Path
import shutil
from google.colab import files

RUN_DIR = Path(
    "/content/drive/MyDrive/fire_smoke_project/"
    "runs/yolov8s_640_100e_b16_s42"
)

assert RUN_DIR.exists(), f"找不到训练结果：{RUN_DIR}"

ZIP_BASE = Path(
    "/content/yolov8s_640_100e_b16_s42_training_results"
)

ZIP_PATH = ZIP_BASE.with_suffix(".zip")

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

shutil.make_archive(
    base_name=str(ZIP_BASE),
    format="zip",
    root_dir=str(RUN_DIR.parent),
    base_dir=RUN_DIR.name
)

print("完整训练结果压缩完成：")
print(ZIP_PATH)
print(
    "压缩包大小：",
    round(ZIP_PATH.stat().st_size / 1024 / 1024, 2),
    "MB"
)

files.download(str(ZIP_PATH))